# Mutual Information for Experimental Data

## Import Packages

In [ ]:
import scanpy as sc
import numpy as np
import pandas as pd
import sys
import torch
import random

from sklearn.preprocessing import StandardScaler

sys.path.append("../../src/discover/")
from metrics import MINE

## Helper Functions

In [ ]:
def compute_mi_embeddings(adata_emb, emb_z, emb_w, hidden_dims, lr, steps, verbose, run_shuffle=False, normalize=False):
    """
    Compute MI (MINE) between two embeddings stored in adata_emb.obsm.
    Optionally computes MI with shuffled second embedding as a control.
    Returns mean and std of MI estimates across multiple seeds.

    Parameters:
    ----------
    adata_emb: AnnData
        AnnData object containing the embeddings.
    emb_z: str
        Key for the first embedding in adata_emb.obsm.
    emb_w: str
        Key for the second embedding in adata_emb.obsm.
    hidden_dims: list of int
        List of hidden layer dimensions for the MINE network.
    lr: float
        Learning rate for MINE.
    steps: int
        Number of training steps for MINE.
    verbose: bool
        Whether to print training progress.
    run_shuffle: bool, optional (default: False)
        Whether to compute MI with shuffled second embedding as a control.
    normalize: bool, optional (default: False)
        Whether to normalize mutual information estimates.

    Returns:
    -------
    dict with keys:
        - "mi_mean": mean MI estimate across seeds
        - "mi_std": std of MI estimates across seeds
        - "shuffle_mean": mean MI estimate with shuffled control (NaN if run_shuffle is False)
        - "shuffle_std": std of MI estimates with shuffled control (NaN if run_shuffle is False)
    """
    # validate keys
    if emb_z not in adata_emb.obsm_keys():
        raise KeyError(f"{emb_z} not found in adata_emb.obsm")
    if emb_w not in adata_emb.obsm_keys():
        raise KeyError(f"{emb_w} not found in adata_emb.obsm")
    
    # get embeddings
    z = np.asarray(adata_emb.obsm[emb_z])
    w = np.asarray(adata_emb.obsm[emb_w])

    # check shapes
    if z.shape[0] != w.shape[0]:
        raise ValueError(f"Number of cells mismatch: {emb_z} has {z.shape[0]} rows, {emb_w} has {w.shape[0]} rows")
    
    # handle NaNs / infs
    if not np.isfinite(z).all() or not np.isfinite(w).all():
        raise ValueError("Embeddings contain NaN or infinite values — clean or impute before MI estimation")

    # scale embeddings
    scaler_z = StandardScaler()
    scaler_w = StandardScaler()
    z_scaled = scaler_z.fit_transform(z)
    w_scaled = scaler_w.fit_transform(w)

    # enforce dtype
    z_scaled = z_scaled.astype(np.float32)
    w_scaled = w_scaled.astype(np.float32)

    # compute normalizer if requested (Gaussian entropy approximation)
    if normalize:
        d_z = z_scaled.shape[1]
        d_w = w_scaled.shape[1]
        # For standardized Gaussian: H(X) ≈ 0.5 * d * (1 + log(2π))
        h_z = 0.5 * d_z * (1.0 + np.log(2.0 * np.pi))
        h_w = 0.5 * d_w * (1.0 + np.log(2.0 * np.pi))
        normalizer = 0.5 * (h_z + h_w)  # arithmetic mean
    else:
        normalizer = 1.0

    # device
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # reproducibility helper
    seeds = [0, 42, 7, 1234, 2021]
    mi_results = []
    mi_shuffle_results = []

    # main loop over seeds
    for seed in seeds:
        np.random.seed(seed)
        random.seed(seed)
        torch.manual_seed(seed)
        if device.type == "cuda":
            torch.cuda.manual_seed_all(seed)
            torch.backends.cudnn.deterministic = True
            torch.backends.cudnn.benchmark = False

        # move tensors to device
        z_tensor = torch.from_numpy(z_scaled).to(device)
        w_tensor = torch.from_numpy(w_scaled).to(device)

        # check dims
        if z_tensor.ndim != 2 or w_tensor.ndim != 2:
            raise ValueError("Embeddings must be 2D (n_samples x dim) after preprocessing")

        # calculate input dim for MINE
        in_dim = int(z_tensor.shape[1] + w_tensor.shape[1])
        if in_dim <= 0:
            raise ValueError("Computed input dimension for MINE is non-positive")
        
        # run MINE
        mine = MINE(in_dim=in_dim, hidden_dims=hidden_dims, lr=lr, verbose=verbose)
        mi = mine.mutual_information(z_tensor, w_tensor, steps=steps)
        mi_results.append(float(mi) / normalizer)

        # run shuffled control if requested
        if run_shuffle:
            # reproducible permutation on the device
            gen = torch.Generator(device=device).manual_seed(seed)
            perm = torch.randperm(w_tensor.size(0), generator=gen, device=device)
            w_shuffled = w_tensor[perm]
            mine_sh = MINE(in_dim=in_dim, hidden_dims=hidden_dims, lr=lr, verbose=verbose)
            mi_shuffle = mine_sh.mutual_information(z_tensor, w_shuffled, steps=steps)
            mi_shuffle_results.append(float(mi_shuffle) / normalizer)

    # aggregate
    mi_mean, mi_std = float(np.mean(mi_results)), float(np.std(mi_results))
    if run_shuffle:
        shuffle_mean, shuffle_std = float(np.mean(mi_shuffle_results)), float(np.std(mi_shuffle_results))
    else:
        shuffle_mean, shuffle_std = float(np.nan), float(np.nan)

    return {
        "mi_mean": float(mi_mean),
        "mi_std": float(mi_std),
        "shuffle_mean": float(shuffle_mean),
        "shuffle_std": float(shuffle_std),
    }

In [ ]:
def get_selection_pca_embeddings(adata_sel, selection_name, n_comps=10, layer="logcounts"):
    """
    Get PCA embeddings for a specific selection.
    Parameters:
    ----------
    adata_sel: AnnData
        AnnData object containing the selections.
    selection_name: str
        Name of the selection to extract.
    n_comps: int, optional (default: 5)
        Number of PCA components to compute.
    layer: str or None, optional (default: "logcounts")
        Layer in adata_sel to use for PCA. If None, uses adata_sel.X

    Returns:
    -------
    X_pca : np.ndarray, shape (n_obs, n_returned_comps)
        PCA embedding matrix for the selected genes. dtype float32.
        Note: the number of returned components may be smaller than n_comps when data
        dimensions limit PCA.
    """
    # validate selection_name exists in var
    if selection_name not in adata_sel.var_keys():
        raise KeyError(f"{selection_name} not found in adata_sel.var")
    
    # validate selection is not empty
    mask = np.asarray(adata_sel.var[selection_name].to_list(), dtype=bool)
    if mask.sum() == 0:
        raise ValueError(f"Selection '{selection_name}' is empty (no genes selected)")

    # filter adata by selection
    new_adata = adata_sel[:, mask].copy()

    # determine valid number of components
    max_comps = min(n_comps, new_adata.n_obs - 1, new_adata.n_vars)
    if max_comps <= 0:
        raise ValueError(f"Not enough observations/variables for PCA on selection '{selection_name}'")
    
    # require layer exists if specified
    if layer is not None and layer not in new_adata.layers:
        raise KeyError(f"Layer '{layer}' not found in adata (available layers: {list(new_adata.layers.keys())})")

    # compute PCA for selection
    sc.pp.pca(new_adata, n_comps=max_comps, layer=layer, random_state=0)
    X_pca = new_adata.obsm["X_pca"]

    # ensure numpy array, contiguous and float32 for downstream code
    X_pca = np.asarray(X_pca)
    X_pca = np.ascontiguousarray(X_pca, dtype=np.float32)

    return X_pca


def compute_mi_selections(adata_sel, selections, hidden_dims, lr, steps, verbose, run_shuffle=False, normalize=False):
    """
    Compute MI (MINE) between multiple selections stored in adata_sel.var.
    Optionally computes MI with shuffled second selection as a control.
    Returns mean and std of MI estimates across multiple seeds for each selection pair.

    Parameters:
    ----------
    adata_sel: AnnData
        AnnData object containing the selections.
    selections: list of tuples
        List of (type_selection, state_selection) tuples.
    hidden_dims: list of int
        List of hidden layer dimensions for the MINE network.
    lr: float
        Learning rate for MINE.
    steps: int
        Number of training steps for MINE.
    verbose: bool
        Whether to print training progress.
    run_shuffle: bool, optional (default: False)
        Whether to compute MI with shuffled control for each selection pair.
    normalize: bool, optional (default: False)
        Whether to normalize mutual information estimates.

    Returns:
    -------
    dict with keys for each selection pair (e.g., "type_state") and values:
        - "mi_mean": mean MI estimate across seeds
        - "mi_std": std of MI estimates across seeds
        - "shuffle_mean": mean MI estimate with shuffled control (NaN if run_shuffle is False)
        - "shuffle_std": std of MI estimates with shuffled control (NaN if run_shuffle is False)
    """
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    results = {}

    # calculate MI for each type and corresponding state selection
    for sel in selections:

        # get selection names
        t = sel[0]
        s = sel[1]

        # get PCA embeddings for selections
        t_pca = get_selection_pca_embeddings(adata_sel, t)
        s_pca = get_selection_pca_embeddings(adata_sel, s)

        # convert to numpy arrays
        t_arr = np.asarray(t_pca, dtype=np.float32)
        s_arr = np.asarray(s_pca, dtype=np.float32)

        # scale embeddings
        scaler_t = StandardScaler()
        scaler_s = StandardScaler()
        t_arr = scaler_t.fit_transform(t_arr)
        s_arr = scaler_s.fit_transform(s_arr)

        # quick shape check
        if t_arr.shape[0] != s_arr.shape[0]:
            raise ValueError(f"Row mismatch for {t} vs {s}: {t_arr.shape[0]} vs {s_arr.shape[0]}")
        
        # compute normalizer if requested
        if normalize:
            d_t = t_arr.shape[1]
            d_s = s_arr.shape[1]
            h_t = 0.5 * d_t * (1.0 + np.log(2.0 * np.pi))
            h_s = 0.5 * d_s * (1.0 + np.log(2.0 * np.pi))
            normalizer = 0.5 * (h_t + h_s)
        else:
            normalizer = 1.0
        
        # convert to torch tensors
        t_tensor = torch.from_numpy(t_arr).to(device)
        s_tensor = torch.from_numpy(s_arr).to(device)

        # set parameters for MI estimation
        in_dim = t_tensor.shape[1] + s_tensor.shape[1]
        seeds = [0, 42, 7, 1234, 2021]
        mi_results = []
        mi_shuffle_results = []

        # compute MI across seeds
        for seed in seeds:
            # set seeds
            np.random.seed(seed)
            random.seed(seed)
            torch.manual_seed(seed)
            if device.type == "cuda":
                torch.cuda.manual_seed_all(seed)
                torch.backends.cudnn.deterministic = True
                torch.backends.cudnn.benchmark = False

            # run MINE
            mine = MINE(in_dim=in_dim, hidden_dims=hidden_dims, lr=lr, verbose=verbose)
            mi = mine.mutual_information(t_tensor, s_tensor, steps=steps)
            mi_results.append(float(mi) / normalizer)

            # run shuffled control if requested
            if run_shuffle:
                gen = torch.Generator(device=device)
                gen.manual_seed(seed)
                perm = torch.randperm(s_tensor.size(0), generator=gen, device=device)
                s_shuffled = s_tensor[perm]
                mine_sh = MINE(in_dim=in_dim, hidden_dims=hidden_dims, lr=lr, verbose=verbose)
                mi_shuffle = mine_sh.mutual_information(t_tensor, s_shuffled, steps=steps)
                mi_shuffle_results.append(float(mi_shuffle) / normalizer)

        results[f"{t}_{s}"] = {
            "mi_mean": float(np.mean(mi_results)),
            "mi_std": float(np.std(mi_results)),
            "shuffle_mean": float(np.mean(mi_shuffle_results)) if run_shuffle else np.nan,
            "shuffle_std": float(np.std(mi_shuffle_results)) if run_shuffle else np.nan,
        }

    return results

## Load and Prepare Data

In [ ]:
# Patches embeddings
patches_emb_clu_con = sc.read_h5ad("../../data/kang/04-emb/Kang18-patches_clu_con.h5ad")
patches_emb_con = sc.read_h5ad("../../data/kang/04-emb/Kang18-patches_con.h5ad")

# DISCoVeR embeddings
discover_emb_con = sc.read_h5ad("../../data/kang/04-emb/Kang18-discover.h5ad")

# selections
patches_sel_clu_con = sc.read_h5ad("../../data/kang/03-sel/Kang18-selections_clu_con.h5ad")
patches_sel_con = sc.read_h5ad("../../data/kang/03-sel/Kang18-selections_con.h5ad")

In [ ]:
selections_clu_con = [
    ("tMAD_sAD", "sAD_tMAD"),
    ("tMAV_sMAV", "sMAV_tMAV"),
    ("tEN_sEN", "sEN_tEN"),
    ("tF_sPBDS", "sPBDS_tF"),
    ("tF_sPVE", "sPVE_tF"),
    ("tPVE_sPVE", "sPVE_tPVE"),
]

selections_con = [
    ("common_sAD", "sAD_common"),
    ("common_sMAV", "sMAV_common"),
    ("common_sEN", "sEN_common"),
    ("tF_sPBDS", "sPBDS_tF"),
    ("tF_sPVE", "sPVE_tF"),
    ("tPVE_sPVE", "sPVE_tPVE"),
]

## Groups & Conditions

In [ ]:
emb_clu_con = compute_mi_embeddings(
    patches_emb_clu_con,
    "patches_z_latent",
    "patches_w_latent",
    hidden_dims=[128, 64],
    lr=1e-3,
    steps=200,
    verbose=True,
    run_shuffle=True,
    normalize=True,
)

emb_clu_con

In [ ]:
sel_clu_con = compute_mi_selections(
    patches_sel_clu_con,
    selections_clu_con,
    hidden_dims=[128, 64],
    lr=1e-3,
    steps=200,
    verbose=True,
    run_shuffle=True,
    normalize=True,
)

sel_clu_con

## Conditions

In [ ]:
emb_con = compute_mi_embeddings(
    patches_emb_con,
    "patches_z_latent",
    "patches_w_latent",
    hidden_dims=[128, 64],
    lr=1e-3,
    steps=200,
    verbose=True,
    run_shuffle=True,
    normalize=True,
)

emb_con

In [ ]:
emb_con_dis = compute_mi_embeddings(
    discover_emb_con,
    "z_discover",
    "w_discover",
    hidden_dims=[128, 64],
    lr=1e-3,
    steps=200,
    verbose=True,
    run_shuffle=True,
    normalize=True,
)

emb_con_dis

In [ ]:
sel_con = compute_mi_selections(
    patches_sel_con,
    selections_con,
    hidden_dims=[128, 64],
    lr=1e-3,
    steps=200,
    verbose=True,
    run_shuffle=True,
    normalize=True,
)

sel_con

## Combine Results and Export

In [ ]:
path = "../../data/kang/05-da/"

### Embeddings

In [ ]:
emb_dicts = {
    "emb_clu_con": emb_clu_con,
    "emb_con": emb_con,
    "emb_con_dis": emb_con_dis,
}

emb_df = pd.DataFrame(emb_dicts).T
emb_df.index.name = "experiment"
emb_df.reset_index(inplace=True)
emb_df

In [ ]:
# export results to csv
emb_df.to_csv(path + "embedding_mi.csv", index=False)

### Selections

In [ ]:
sel_dicts = {
    "sel_clu_con": sel_clu_con,
    "sel_con": sel_con,
}

# flatten nested dicts into rows
rows = []
for exp_name, selections in sel_dicts.items():
    for selection_pair, metrics in selections.items():
        row = {
            "experiment": exp_name,
            "selection_pair": selection_pair,
            **metrics  # unpack mi_mean, mi_std, shuffle_mean, shuffle_std
        }
        rows.append(row)

sel_df = pd.DataFrame(rows)
sel_df

In [ ]:
# export results to csv
sel_df.to_csv(path + "selection_mi.csv", index=False)